In [1]:
import numpy as np

In [2]:
class RevisedSimplex:
    def __init__(self, A, b, c):
        self.A = A
        self.b = b
        self.c = c

        self.m, self.n = A.shape

        self.B_idx = list(range(self.n - self.m, self.n))
        self.N_idx = list(range(self.n - self.m))

    def solve(self):
        while True:
            # --- Step 1 ---
            B = self.A[:, self.B_idx]
            N = self.A[:, self.N_idx]

            B_inv = np.linalg.inv(B)

            x_B = B_inv @ self.b  

            # --- Step 2 ---
            c_B = self.c[self.B_idx]
            c_N = self.c[self.N_idx]

            c_bar = c_N - N.T @ B_inv.T @ c_B   # 被約費用

            # --- Step 3 ---
            if np.all(c_bar <= 0):
                # 最適
                x = np.zeros(self.n)
                x[self.B_idx] = x_B
                return x, self.c @ x

            # entering variable
            k = np.argmax(c_bar)
            entering = self.N_idx[k]

            # --- Step 4 ---
            a_k = self.A[:, entering]
            a_bar = B_inv @ a_k

            if np.all(a_bar <= 0):
                raise Exception("非有界です")

            # --- Step 5 ---
            ratios = []
            for i in range(self.m):
                if a_bar[i] > 0:
                    ratios.append(x_B[i] / a_bar[i])
                else:
                    ratios.append(np.inf)

            i_star = np.argmin(ratios)
            leaving = self.B_idx[i_star]

            # 基底更新
            self.B_idx[i_star] = entering
            self.N_idx[k] = leaving

In [3]:
A = np.array([
    [1, 1, 1, 0, 0],
    [1, 3, 0, 1, 0],
    [2, 1, 0, 0, 1]
], dtype=float)

b = np.array([6, 12, 10], dtype=float)
c = np.array([1, 2, 0, 0, 0], dtype=float)

In [15]:
A = np.array([
    [1, -2, 1, 0],
    [-1, 1, 0, 1]
], dtype=float)

b = np.array([4, 2], dtype=float)
c = np.array([2, 1, 0, 0], dtype=float)

In [4]:
solver = RevisedSimplex(A, b, c)
x_opt, val = solver.solve()

In [5]:
print("x* =", x_opt)
print("optimal value =", val)

x* = [3. 3. 0. 0. 1.]
optimal value = 9.0
